# ResNet-50 Multi-Output Classification with AutoAugment

Implementation of best pipeline from multilabel fashion classification:
- ResNet-50 pretrained model
- Multi-output architecture (jenis & warna)
- AutoAugment for data augmentation
- Exact Match Ratio (EMR) evaluation

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchvision tqdm scikit-learn

## 2. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.io import read_image
from torchvision.transforms import AutoAugment
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 3. Dataset Class with AutoAugment

In [ ]:
class MultiValuedDataset(torch.utils.data.Dataset):
    def __init__(self, df_path, img_path, transform=None, is_train=True):
        self.df = pd.read_csv(df_path, index_col='id', sep=',')
        self.transform = transform
        self.img_path = img_path
        self.auto_augment = AutoAugment()
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Find image file
        img_file = None
        for ext in ['.jpg', '.png']:
            img_path = os.path.join(self.img_path, f'{self.df.index[idx]}{ext}')
            if os.path.exists(img_path):
                img_file = img_path
                break
        
        if not img_file:
            raise FileNotFoundError(f"Image {self.df.index[idx]} not found")

        img = read_image(img_file)

        # Apply AutoAugment only for training
        if self.is_train:
            img = self.auto_augment(img)

        if self.transform:
            img = self.transform(img)

        jenis_label = self.df.iloc[idx]['jenis']
        warna_label = self.df.iloc[idx]['warna']

        return img, jenis_label, warna_label

## 4. Data Preprocessing and Loading

In [ ]:
# Transform pipeline
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load datasets
train_data = MultiValuedDataset(
    df_path='train.csv',
    img_path='train/train',
    transform=transform,
    is_train=True
)

val_data = MultiValuedDataset(
    df_path='train.csv',
    img_path='train/train',
    transform=transform,
    is_train=False
)

# Split indices for train/val
train_indices = list(range(len(train_data)))
train_idx, val_idx = train_test_split(
    train_indices, 
    test_size=0.2, 
    random_state=42,
    stratify=[train_data[i][1] for i in train_indices]
)

# Create subset datasets
train_subset = torch.utils.data.Subset(train_data, train_idx)
val_subset = torch.utils.data.Subset(val_data, val_idx)

# DataLoaders (num_workers=0 to avoid multiprocessing issues in notebooks)
train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=0)

print(f'Training samples: {len(train_subset)}')
print(f'Validation samples: {len(val_subset)}')

## 5. ResNet-50 Multi-Output Model

In [ ]:
class ResNetMultiOutputModel(nn.Module):
    def __init__(self, num_classes_jenis=2, num_classes_warna=5):
        super(ResNetMultiOutputModel, self).__init__()
        
        # Load pretrained ResNet-50
        resnet = models.resnet50(pretrained=True)
        
        # Feature extractor (all layers except final FC)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        
        # Get number of features
        num_features = resnet.fc.in_features
        
        # Separate FC layers for each task
        self.fc_jenis = nn.Linear(num_features, num_classes_jenis)
        self.fc_warna = nn.Linear(num_features, num_classes_warna)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        
        jenis_output = self.fc_jenis(x)
        warna_output = self.fc_warna(x)
        
        return [jenis_output, warna_output]

# Initialize model
model = ResNetMultiOutputModel().to(device)
print('Model initialized')

## 6. Training Configuration

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)
num_epochs = 50

print(f'Optimizer: AdamW (lr=0.0001, weight_decay=1e-5)')
print(f'Loss function: CrossEntropyLoss')
print(f'Epochs: {num_epochs}')

## 7. Training Loop

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    
    loop = tqdm(dataloader, leave=False)
    for images, jenis_labels, warna_labels in loop:
        images = images.to(device)
        jenis_labels = jenis_labels.to(device)
        warna_labels = warna_labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss_jenis = criterion(outputs[0], jenis_labels)
        loss_warna = criterion(outputs[1], warna_labels)
        loss = (loss_jenis + loss_warna) / 2
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    
    return running_loss / len(dataloader)

# Training
train_losses = []

for epoch in range(num_epochs):
    avg_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

print('Training completed')

## 8. Validation with Exact Match Ratio

In [ ]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    exact_matches = 0
    total = 0
    
    with torch.no_grad():
        for images, jenis_labels, warna_labels in dataloader:
            images = images.to(device)
            jenis_labels = jenis_labels.to(device)
            warna_labels = warna_labels.to(device)
            
            outputs = model(images)
            loss_jenis = criterion(outputs[0], jenis_labels)
            loss_warna = criterion(outputs[1], warna_labels)
            loss = (loss_jenis + loss_warna) / 2
            
            running_loss += loss.item()
            
            # Get predictions
            _, jenis_preds = torch.max(outputs[0], 1)
            _, warna_preds = torch.max(outputs[1], 1)
            
            # Calculate exact matches
            matches = ((jenis_preds == jenis_labels) & (warna_preds == warna_labels))
            exact_matches += matches.sum().item()
            total += jenis_labels.size(0)
    
    avg_loss = running_loss / len(dataloader)
    emr = (exact_matches / total) * 100
    
    return avg_loss, emr

# Validate
val_loss, emr = validate(model, val_loader, criterion, device)
print(f'Validation Loss: {val_loss:.4f}')
print(f'Exact Match Ratio (EMR): {emr:.2f}%')

## 9. Test Prediction

In [ ]:
class TestDataset(torch.utils.data.Dataset):
    def __init__(self, img_path, transform=None):
        self.img_path = img_path
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(img_path) if f.endswith(('.jpg', '.png'))])
        
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_file = os.path.join(self.img_path, self.image_files[idx])
        img = read_image(img_file)
        
        if self.transform:
            img = self.transform(img)
        
        img_id = int(self.image_files[idx].split('.')[0])
        return img, img_id

# Load test data
test_data = TestDataset('test/test', transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Predict
model.eval()
predictions = []

with torch.no_grad():
    for images, img_ids in tqdm(test_loader):
        images = images.to(device)
        outputs = model(images)
        
        _, jenis_preds = torch.max(outputs[0], 1)
        _, warna_preds = torch.max(outputs[1], 1)
        
        for img_id, jenis, warna in zip(img_ids, jenis_preds, warna_preds):
            predictions.append({
                'id': img_id.item(),
                'jenis': jenis.item(),
                'warna': warna.item()
            })

# Create submission
submission = pd.DataFrame(predictions)
submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv('submission_resnet50.csv', index=False)

print(f'Predictions saved to submission_resnet50.csv')
print(f'Total predictions: {len(submission)}')

## 10. Save Model

In [ ]:
torch.save(model.state_dict(), 'resnet50_multioutput.pth')
print('Model saved to resnet50_multioutput.pth')

## 11. Separate Pretrained Models (Ensemble Approach)

In [ ]:
class SingleTaskModel(nn.Module):
    def __init__(self, num_classes):
        super(SingleTaskModel, self).__init__()
        resnet = models.resnet50(pretrained=True)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        num_features = resnet.fc.in_features
        self.fc = nn.Linear(num_features, num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# Initialize separate models
model_jenis = SingleTaskModel(num_classes=2).to(device)
model_warna = SingleTaskModel(num_classes=5).to(device)

print('Separate models initialized')
print(f'Jenis model: 2 classes')
print(f'Warna model: 5 classes')

### 11.1 Train Jenis Model

In [ ]:
criterion_jenis = nn.CrossEntropyLoss()
optimizer_jenis = optim.AdamW(model_jenis.parameters(), lr=0.0001, weight_decay=1e-5)

def train_single_task(model, dataloader, criterion, optimizer, device, task='jenis'):
    model.train()
    running_loss = 0.0
    
    for images, jenis_labels, warna_labels in tqdm(dataloader, leave=False):
        images = images.to(device)
        labels = jenis_labels.to(device) if task == 'jenis' else warna_labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    return running_loss / len(dataloader)

# Train jenis model
print('Training Jenis model...')
for epoch in range(30):
    loss = train_single_task(model_jenis, train_loader, criterion_jenis, optimizer_jenis, device, 'jenis')
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/30], Loss: {loss:.4f}')

print('Jenis model training completed')

### 11.2 Train Warna Model

In [ ]:
criterion_warna = nn.CrossEntropyLoss()
optimizer_warna = optim.AdamW(model_warna.parameters(), lr=0.0001, weight_decay=1e-5)

# Train warna model
print('Training Warna model...')
for epoch in range(30):
    loss = train_single_task(model_warna, train_loader, criterion_warna, optimizer_warna, device, 'warna')
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/30], Loss: {loss:.4f}')

print('Warna model training completed')

### 11.3 Ensemble Validation

In [ ]:
def validate_ensemble(model_jenis, model_warna, dataloader, device):
    model_jenis.eval()
    model_warna.eval()
    
    exact_matches = 0
    total = 0
    
    with torch.no_grad():
        for images, jenis_labels, warna_labels in dataloader:
            images = images.to(device)
            jenis_labels = jenis_labels.to(device)
            warna_labels = warna_labels.to(device)
            
            # Get predictions from both models
            jenis_outputs = model_jenis(images)
            warna_outputs = model_warna(images)
            
            _, jenis_preds = torch.max(jenis_outputs, 1)
            _, warna_preds = torch.max(warna_outputs, 1)
            
            # Calculate exact matches
            matches = ((jenis_preds == jenis_labels) & (warna_preds == warna_labels))
            exact_matches += matches.sum().item()
            total += jenis_labels.size(0)
    
    emr = (exact_matches / total) * 100
    return emr

# Validate ensemble
emr_ensemble = validate_ensemble(model_jenis, model_warna, val_loader, device)
print(f'Ensemble Exact Match Ratio (EMR): {emr_ensemble:.2f}%')

### 11.4 Ensemble Test Prediction

In [ ]:
model_jenis.eval()
model_warna.eval()
predictions_ensemble = []

with torch.no_grad():
    for images, img_ids in tqdm(test_loader):
        images = images.to(device)
        
        # Get predictions from both models
        jenis_outputs = model_jenis(images)
        warna_outputs = model_warna(images)
        
        _, jenis_preds = torch.max(jenis_outputs, 1)
        _, warna_preds = torch.max(warna_outputs, 1)
        
        for img_id, jenis, warna in zip(img_ids, jenis_preds, warna_preds):
            predictions_ensemble.append({
                'id': img_id.item(),
                'jenis': jenis.item(),
                'warna': warna.item()
            })

# Create ensemble submission
submission_ensemble = pd.DataFrame(predictions_ensemble)
submission_ensemble = submission_ensemble.sort_values('id').reset_index(drop=True)
submission_ensemble.to_csv('submission_ensemble.csv', index=False)

print(f'Ensemble predictions saved to submission_ensemble.csv')
print(f'Total predictions: {len(submission_ensemble)}')

### 11.5 Save Ensemble Models

In [ ]:
torch.save(model_jenis.state_dict(), 'resnet50_jenis.pth')
torch.save(model_warna.state_dict(), 'resnet50_warna.pth')
print('Ensemble models saved')
print('- resnet50_jenis.pth')
print('- resnet50_warna.pth')

## 12. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Multi-Output ResNet-50', 'Ensemble (Separate Models)'],
    'EMR (%)': [emr, emr_ensemble],
    'Approach': ['Single model with 2 outputs', '2 separate specialized models']
})

print('Performance Comparison:')
print(comparison.to_string(index=False))
print(f'\nBest model: {comparison.loc[comparison["EMR (%)"].idxmax(), "Model"]}')

## 13. Diverse Pretrained Models Ensemble

In [ ]:
class EfficientNetModel(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetModel, self).__init__()
        efficientnet = models.efficientnet_b0(pretrained=True)
        self.feature_extractor = efficientnet.features
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        num_features = efficientnet.classifier[1].in_features
        self.fc = nn.Linear(num_features, num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

class MobileNetModel(nn.Module):
    def __init__(self, num_classes):
        super(MobileNetModel, self).__init__()
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.feature_extractor = mobilenet.features
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        num_features = mobilenet.classifier[1].in_features
        self.fc = nn.Linear(num_features, num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

# Initialize diverse models for jenis
efficientnet_jenis = EfficientNetModel(num_classes=2).to(device)
mobilenet_jenis = MobileNetModel(num_classes=2).to(device)

# Initialize diverse models for warna
efficientnet_warna = EfficientNetModel(num_classes=5).to(device)
mobilenet_warna = MobileNetModel(num_classes=5).to(device)

print('Diverse models initialized:')
print('- EfficientNet-B0 for Jenis and Warna')
print('- MobileNet-V2 for Jenis and Warna')

### 13.1 Train EfficientNet Models

In [ ]:
# Train EfficientNet Jenis
print('Training EfficientNet-B0 for Jenis...')
optimizer_eff_jenis = optim.AdamW(efficientnet_jenis.parameters(), lr=0.0001, weight_decay=1e-5)
for epoch in range(30):
    loss = train_single_task(efficientnet_jenis, train_loader, criterion_jenis, optimizer_eff_jenis, device, 'jenis')
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/30], Loss: {loss:.4f}')

# Train EfficientNet Warna
print('Training EfficientNet-B0 for Warna...')
optimizer_eff_warna = optim.AdamW(efficientnet_warna.parameters(), lr=0.0001, weight_decay=1e-5)
for epoch in range(30):
    loss = train_single_task(efficientnet_warna, train_loader, criterion_warna, optimizer_eff_warna, device, 'warna')
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/30], Loss: {loss:.4f}')

print('EfficientNet models training completed')

### 13.2 Train MobileNet Models

In [ ]:
# Train MobileNet Jenis
print('Training MobileNet-V2 for Jenis...')
optimizer_mob_jenis = optim.AdamW(mobilenet_jenis.parameters(), lr=0.0001, weight_decay=1e-5)
for epoch in range(30):
    loss = train_single_task(mobilenet_jenis, train_loader, criterion_jenis, optimizer_mob_jenis, device, 'jenis')
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/30], Loss: {loss:.4f}')

# Train MobileNet Warna
print('Training MobileNet-V2 for Warna...')
optimizer_mob_warna = optim.AdamW(mobilenet_warna.parameters(), lr=0.0001, weight_decay=1e-5)
for epoch in range(30):
    loss = train_single_task(mobilenet_warna, train_loader, criterion_warna, optimizer_mob_warna, device, 'warna')
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/30], Loss: {loss:.4f}')

print('MobileNet models training completed')

### 13.3 Multi-Model Ensemble with Voting

In [ ]:
def voting_ensemble_validate(jenis_models, warna_models, dataloader, device):
    for model in jenis_models + warna_models:
        model.eval()
    
    exact_matches = 0
    total = 0
    
    with torch.no_grad():
        for images, jenis_labels, warna_labels in dataloader:
            images = images.to(device)
            jenis_labels = jenis_labels.to(device)
            warna_labels = warna_labels.to(device)
            
            # Collect predictions from all jenis models
            jenis_votes = []
            for model in jenis_models:
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                jenis_votes.append(preds)
            
            # Collect predictions from all warna models
            warna_votes = []
            for model in warna_models:
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                warna_votes.append(preds)
            
            # Majority voting
            jenis_votes = torch.stack(jenis_votes)
            warna_votes = torch.stack(warna_votes)
            
            jenis_preds = torch.mode(jenis_votes, dim=0)[0]
            warna_preds = torch.mode(warna_votes, dim=0)[0]
            
            # Calculate exact matches
            matches = ((jenis_preds == jenis_labels) & (warna_preds == warna_labels))
            exact_matches += matches.sum().item()
            total += jenis_labels.size(0)
    
    emr = (exact_matches / total) * 100
    return emr

# Validate multi-model ensemble
jenis_models = [model_jenis, efficientnet_jenis, mobilenet_jenis]
warna_models = [model_warna, efficientnet_warna, mobilenet_warna]

emr_voting = voting_ensemble_validate(jenis_models, warna_models, val_loader, device)
print(f'Multi-Model Ensemble (Voting) EMR: {emr_voting:.2f}%')

### 13.4 Multi-Model Ensemble Test Prediction

In [ ]:
for model in jenis_models + warna_models:
    model.eval()

predictions_voting = []

with torch.no_grad():
    for images, img_ids in tqdm(test_loader):
        images = images.to(device)
        
        # Collect predictions from all jenis models
        jenis_votes = []
        for model in jenis_models:
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            jenis_votes.append(preds)
        
        # Collect predictions from all warna models
        warna_votes = []
        for model in warna_models:
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            warna_votes.append(preds)
        
        # Majority voting
        jenis_votes = torch.stack(jenis_votes)
        warna_votes = torch.stack(warna_votes)
        
        jenis_preds = torch.mode(jenis_votes, dim=0)[0]
        warna_preds = torch.mode(warna_votes, dim=0)[0]
        
        for img_id, jenis, warna in zip(img_ids, jenis_preds, warna_preds):
            predictions_voting.append({
                'id': img_id.item(),
                'jenis': jenis.item(),
                'warna': warna.item()
            })

# Create voting ensemble submission
submission_voting = pd.DataFrame(predictions_voting)
submission_voting = submission_voting.sort_values('id').reset_index(drop=True)
submission_voting.to_csv('submission_voting_ensemble.csv', index=False)

print(f'Voting ensemble predictions saved to submission_voting_ensemble.csv')
print(f'Total predictions: {len(submission_voting)}')

### 13.5 Final Model Comparison

In [ ]:
final_comparison = pd.DataFrame({
    'Model': [
        'Multi-Output ResNet-50',
        'ResNet-50 Ensemble',
        'Multi-Model Voting Ensemble'
    ],
    'Architecture': [
        'ResNet-50 (1 model, 2 outputs)',
        'ResNet-50 (2 models)',
        'ResNet-50 + EfficientNet-B0 + MobileNet-V2 (6 models)'
    ],
    'EMR (%)': [
        emr,
        emr_ensemble,
        emr_voting
    ]
})

print('Final Performance Comparison:')
print(final_comparison.to_string(index=False))
print(f'\nBest model: {final_comparison.loc[final_comparison["EMR (%)"].idxmax(), "Model"]}')
print(f'Best EMR: {final_comparison["EMR (%)"].max():.2f}%')